In [1]:
##############
## INITIATE ##
##############

## Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import pickle

#
###

In [46]:
###########################
## SCRIPT CONFIGURATIONS ##
###########################

## Parameters
MODEL_NAME = 'model_vit_icgcc_raw_FTL_0'
PT_METADATA = '/Volumes/Elements/BDI/projects/GSG/metadata/icgcc-relapse/metadata_ICGCCRelapse.csv'
DATASET = 'icgc-c'
#
# MODEL_NAME = 'model_resnet_tcgaoct_raw_FTL_25'
# PT_METADATA = '/Volumes/Elements/BDI/projects/GSG/metadata/tcga-cnv/master-sheet-2025-04-18.csv'
# DATASET = 'tcga-oct'

## Define paths to embeddings
PTI = f'/Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/{DATASET}/{MODEL_NAME}/'
PTO = f'/Volumes/Elements/BDI/projects/GSG/outputs/o3_mil/{DATASET}/{MODEL_NAME}/'
os.makedirs(PTO, exist_ok=True)
PT_EMBEDDINGS = PTI + 'embeddings.npy'
PT_IDENTIFIERS = PTI + 'image_identifiers.csv'

## Define paths to images
PT_DATASET = f'/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/{DATASET}/'
# dataset_path = f'/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/{DATASET}/'
# dataset_path = '/Users/user/Documents/projects/GlandSeg/outputs/V2/o1_extracted_glands/tcga_ffpes_V1'
# dataset_path = '/Users/user/Documents/projects/GlandSeg/outputs/V2_0_0_20250120/o1_extracted_glands/debug'

#
###

In [47]:
######################
## LOAD EMBEDDINGS  ##
######################

## Load the embeddings from the .npy file
embeddings = np.load(PT_EMBEDDINGS)

## Load identifiers
identifiers_df = pd.read_csv(PT_IDENTIFIERS)
image_filenames = identifiers_df['filename'].tolist()

## Check the shape of the loaded embeddings and identifiers
print(f'Loaded embeddings shape: {embeddings.shape}')
print(f'Loaded {len(image_filenames)} image identifiers')

#
###

Loaded embeddings shape: (107199, 512)
Loaded 107199 image identifiers


In [48]:
#######################
## FORMAT IDENTIFIER ##
#######################

## Get unique identifiers
split_marker = '.svs' # For ICGC
# split_marker = '_'
image_filenames_short = [fname.split(split_marker)[1] for fname in image_filenames]
# split_marker = '-01' # For TCGA
image_filenames_short = [fname.split(split_marker)[0] for fname in image_filenames] # For TCGA
# image_filenames_long = [fname.split('.svs')[0].replace('.','-') for fname in image_filenames] # For TCGA
unique_ids = np.unique(image_filenames_short)
num_groups = len(unique_ids)
print(num_groups)
# print(image_filenames_long[0])
numeric_ids = np.zeros(len(image_filenames_short))
k = 0
for unique_id in unique_ids:    
    index = np.argwhere(np.array(image_filenames_short) == unique_id)
    numeric_ids[index] = k
    k += 1

## Checks
print(image_filenames[0:10])
print(image_filenames_short[0:10])
print(numeric_ids[0:10])
print(len(numeric_ids))

#
###

756
['TB08.0618_V5_01.svs__1456_1568_4368_4480_raw.png', 'TB08.0618_V5_01.svs__1488_1648_5184_5344_raw.png', 'TB08.0618_V5_01.svs__1528_1704_4096_4272_raw.png', 'TB08.0618_V5_01.svs__1568_2000_4536_4968_raw.png', 'TB08.0618_V5_01.svs__1664_1968_5008_5312_raw.png', 'TB08.0618_V5_01.svs__1712_2400_4080_4768_raw.png', 'TB08.0618_V5_01.svs__1744_1968_5440_5664_raw.png', 'TB08.0618_V5_01.svs__2016_2080_2240_2304_raw.png', 'TB08.0618_V5_01.svs__2048_2784_4968_5704_raw.png', 'TB08.0618_V5_01.svs__2080_2272_3304_3496_raw.png']
['TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01', 'TB08.0618_V5_01']
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
107199


In [49]:
#####################
## GET COORDINATES ##
#####################

## Get coordinates
split_marker = '__'
coordinates = [fname.split('__')[1] for fname in image_filenames]
coordinates = [fname.split('.')[0] for fname in coordinates]
coordinates = [fname.split('_')[0:4] for fname in coordinates]

## Format
coordinates = np.array(coordinates, dtype=int)

## Checks
print(coordinates[0:10])

#
###

[[1456 1568 4368 4480]
 [1488 1648 5184 5344]
 [1528 1704 4096 4272]
 [1568 2000 4536 4968]
 [1664 1968 5008 5312]
 [1712 2400 4080 4768]
 [1744 1968 5440 5664]
 [2016 2080 2240 2304]
 [2048 2784 4968 5704]
 [2080 2272 3304 3496]]


In [ ]:
###############################
## FORMAT EMBEDDINGS FOR MIL ##
###############################

import numpy as np
import pandas as pd

# Load your metadata CSV (assuming it's called metadata.csv)
df = pd.read_csv(PT_METADATA)  # replace with your actual file path
print(df.shape)

# Ensure slide_id column exists
assert 'sample_id_local' in df.columns, "'sample_id_local' must be a column in your metadata CSV."

# For fast lookup: create a mapping from slide_id to its indices in image_filenames_short
from collections import defaultdict

slideid_to_indices = defaultdict(list)
for idx, name in enumerate(image_filenames_short):
    slideid_to_indices[name].append(idx)

# Assemble per-slide_id embedding subsets
embedding_groups = []
filename_groups = []
coordinate_groups = []
slide_ids_used = []

for slide_id in df['sample_id_local']:
    if slide_id in slideid_to_indices:
        indices = slideid_to_indices[slide_id]
        emb_group = np.array([embeddings[i] for i in indices])
        fil_group = np.array([image_filenames[i] for i in indices])
        coo_group = np.array([coordinates[i] for i in indices])
        embedding_groups.append(emb_group)
        filename_groups.append(fil_group)
        coordinate_groups.append(coo_group)
        slide_ids_used.append(slide_id)
    else:
        # Optional: print or track missing slide_ids
        print(f"missing {slide_id}")
        pass  # or print(f"Slide ID {slide_id} not found in image_filenames_short")

## Save embeddings
os.makedirs(PTO + "encoder/", exist_ok=True)
print(PTO + "encoder/")
np.save(PTO + "encoder/obj_embeddings.npy", np.array(embedding_groups, dtype=object))  # list of arrays
with open(PTO + "encoder/obj_embeddings.pkl", "wb") as f:
    pickle.dump(embedding_groups, f)

## Save filenames with coordinates
os.makedirs(PTO + "preprocessed/", exist_ok=True)
np.save(PTO + "preprocessed/obj_filenames.npy", np.array(filename_groups, dtype=object))  # list of arrays
with open(PTO + "preprocessed/obj_filenames.pkl", "wb") as f:
    pickle.dump(filename_groups, f)

## Save filenames with coordinates
os.makedirs(PTO + "preprocessed/", exist_ok=True)
np.save(PTO + "preprocessed/obj_coordinates.npy", np.array(coordinate_groups, dtype=object))  # list of arrays
with open(PTO + "preprocessed/obj_coordinates.pkl", "wb") as f:
    pickle.dump(coordinate_groups, f)

## Save ids
# np.save(PTO + "obj_slide_ids.npy", np.array(slide_ids_used))
    
#
###